# GOLD LAYER


Transform the Silver Layer dataset into a Star Schema model consisting of Fact and Dimension tables optimized for Power BI reporting and business intelligence analysis.

## Import libraries

In [1]:
import numpy as np
import pandas as pd


## Connect Google Drive
Connecting Google Drive to access the project data and save outputs across all Medallion layers.

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


## Import Silver Layer Data
Load the cleaned and transformed data from the Silver Layer to prepare it for dimensional modeling and analytical reporting.

In [3]:
df = pd.read_csv(
    "/content/drive/MyDrive/Depi Grad Project/Data/Silver/silver_salesss.csv"
)



## Initial Data Inspection


In [4]:
df.head()

,Date,Branch,Drug_Name,Sales_Quantity,Unit_Price,Unit_Cost,Expired_Quantity,Damaged_Quantity,Revenue,Total_Cost,Expired_Loss,Damaged_Loss,Net_Profit,Waste_Quantity,Waste_Rate,Profit_Margin,Year,Month,Month_Name
0,2021-01-01,Branch_01,Paracetamol 500Mg,24,89.82,58.64,0,0,2155.68,1407.36,0.0,0.00,748.32,0,0.000000,0.347139,2021,1,January
1,2021-01-01,Branch_01,Amoxicillin 500Mg,135,100.94,67.55,2,1,13626.90,9119.25,135.1,67.55,4305.00,3,0.022222,0.315919,2021,1,January
2,2021-01-01,Branch_01,Azithromycin 250Mg,552,35.43,20.77,0,3,19557.36,11465.04,0.0,62.31,8030.01,3,0.005435,0.410588,2021,1,January
3,2021-01-01,Branch_01,Metformin 850Mg,38,29.24,21.71,0,0,1111.12,824.98,0.0,0.00,286.14,0,0.000000,0.257524,2021,1,January
4,2021-01-01,Branch_01,Atorvastatin 20Mg,20,87.09,57.26,0,0,1741.80,1145.20,0.0,0.00,596.60,0,0.000000,0.342519,2021,1,January


## Create Product Dimension Table
Create a Product Dimension table containing unique drug names and assign a surrogate ProductKey for each product. This table will be used for product-level analysis and reporting.

In [5]:
dim_product = (
    df[['Drug_Name']]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_product['ProductKey'] = range(1, len(dim_product) + 1)

dim_product = dim_product[
    ['ProductKey', 'Drug_Name']
]

In [6]:
dim_product

,ProductKey,Drug_Name
0,1,Paracetamol 500Mg
1,2,Amoxicillin 500Mg
2,3,Azithromycin 250Mg
3,4,Metformin 850Mg
4,5,Atorvastatin 20Mg
5,6,Aspirin 81Mg
6,7,Ceftriaxone 1G
7,8,Ibuprofen 400Mg
8,9,Omeprazole 20Mg
9,10,Salbutamol Inhaler


## Create Branch Dimension Table


Create a Branch Dimension table containing unique pharmacy branches and assign a surrogate BranchKey for each branch to support branch-level analysis

In [7]:
dim_branch = (
    df[['Branch']]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_branch['BranchKey'] = range(1, len(dim_branch) + 1)

dim_branch = dim_branch[
    ['BranchKey', 'Branch']
]

In [8]:
dim_branch

,BranchKey,Branch
0,1,Branch_01
1,2,Branch_02
2,3,Branch_03
3,4,Branch_04
4,5,Branch_05
5,6,Branch_06
6,7,Branch_07
7,8,Branch_08
8,9,Branch_09
9,10,Branch_10


## Create Date Dimension Table
Build a Date Dimension table from the transaction dates and generate date attributes such as Year, Month, Month Name, and Quarter to enable time-based analysis.

In [9]:
df['Date'] = pd.to_datetime(df['Date'])

dim_date = (
    df[['Date']]
    .drop_duplicates()
    .sort_values('Date')
    .reset_index(drop=True)
)

dim_date['DateKey'] = dim_date['Date'].dt.strftime('%Y%m%d').astype(int)
dim_date['Year'] = dim_date['Date'].dt.year
dim_date['Month'] = dim_date['Date'].dt.month
dim_date['MonthName'] = dim_date['Date'].dt.month_name()
dim_date['Quarter'] = dim_date['Date'].dt.quarter

In [10]:
dim_date

,Date,DateKey,Year,Month,MonthName,Quarter
0,2021-01-01,20210101,2021,1,January,1
1,2021-01-02,20210102,2021,1,January,1
2,2021-01-03,20210103,2021,1,January,1
3,2021-01-04,20210104,2021,1,January,1
4,2021-01-05,20210105,2021,1,January,1
...,...,...,...,...,...,...
1090,2023-12-27,20231227,2023,12,December,4
1091,2023-12-28,20231228,2023,12,December,4
1092,2023-12-29,20231229,2023,12,December,4
1093,2023-12-30,20231230,2023,12,December,4


## Build Fact Table
Create the central Fact Sales table by linking transactional data with Product and Branch dimensions through surrogate keys

In [11]:
fact_sales = df.merge(
    dim_product,
    on='Drug_Name',
    how='left'
)

fact_sales = fact_sales.merge(
    dim_branch,
    on='Branch',
    how='left'
)



## Generate DateKey
Create a DateKey in YYYYMMDD format to establish a relationship between the Fact Table and the Date Dimension

In [12]:

fact_sales['DateKey'] = (
    pd.to_datetime(fact_sales['Date'])
    .dt.strftime('%Y%m%d')
    .astype(int)
)

## Select Fact Table Measures


In [13]:
fact_sales = fact_sales[
[
    'DateKey',
    'ProductKey',
    'BranchKey',

    'Sales_Quantity',
    'Unit_Price',
    'Unit_Cost',

    'Revenue',
    'Total_Cost',
    'Net_Profit',
    'Profit_Margin',

    'Expired_Quantity',
    'Damaged_Quantity',
    'Waste_Quantity',
    'Waste_Rate'




]
]

In [14]:
fact_sales.head()


,DateKey,ProductKey,BranchKey,Sales_Quantity,Unit_Price,Unit_Cost,Revenue,Total_Cost,Net_Profit,Profit_Margin,Expired_Quantity,Damaged_Quantity,Waste_Quantity,Waste_Rate
0,20210101,1,1,24,89.82,58.64,2155.68,1407.36,748.32,0.347139,0,0,0,0.000000
1,20210101,2,1,135,100.94,67.55,13626.90,9119.25,4305.00,0.315919,2,1,3,0.022222
2,20210101,3,1,552,35.43,20.77,19557.36,11465.04,8030.01,0.410588,0,3,3,0.005435
3,20210101,4,1,38,29.24,21.71,1111.12,824.98,286.14,0.257524,0,0,0,0.000000
4,20210101,5,1,20,87.09,57.26,1741.80,1145.20,596.60,0.342519,0,0,0,0.000000


## Export Gold Layer Tables

In [15]:
dim_product.to_csv(
    "/content/drive/MyDrive/Depi Grad Project/Data/Gold/dim_productt.csv",
    index=False
)

dim_branch.to_csv(
    "/content/drive/MyDrive/Depi Grad Project/Data/Gold/dim_branchh.csv",
    index=False
)

dim_date.to_csv(
    "/content/drive/MyDrive/Depi Grad Project/Data/Gold/dim_datee.csv",
    index=False
)

fact_sales.to_csv(
    "/content/drive/MyDrive/Depi Grad Project/Data/Gold/fact_saless.csv",
    index=False
)

